# Open Food Facts — Exploratory Data Analysis

**Project:** Canadian Food Database (OLAP Pipeline)  
**Goal:** Understand data quality, structure, and plan transformations for the ELT pipeline.

---

## Table of Contents
1. [Setup & Data Loading](#1-setup--data-loading)
2. [Data Quality Assessment](#2-data-quality-assessment)
   - 2.1 Schema Overview
   - 2.2 Primary Key Verification
   - 2.3 Duplicates Check
   - 2.4 Null Analysis
3. [Data Profiling](#3-data-profiling)
   - 3.1 Numeric Columns
   - 3.2 String Columns
   - 3.3 Complex Columns (List & Struct)
4. [Canadian Products Analysis](#4-canadian-products-analysis)
5. [Core Analytical Columns](#5-core-analytical-columns)
6. [ELT Action Plan](#6-elt-action-plan)

---
## 1. Setup & Data Loading

In [ ]:
import polars as pl
import polars.selectors as cs
import requests
import openfoodfacts

### Reproducible Sampling via Hash

We use `code.hash()` to create a deterministic random sample.
This ensures the **same 50,000 rows are selected every run**, which is critical for:
- Reproducible EDA results
- Consistent comparisons across team members
- Stable debugging sessions

> **Why not `.limit(50_000)`?**  
> `.limit()` takes the first N rows, which may be biased (e.g., all products from the same brand or country).
> Hash-based sorting distributes rows pseudo-randomly across the full dataset.

In [ ]:
df = (
    pl.scan_parquet("/home/sara/food.parquet")
    .with_columns(pl.col("code").hash().alias("_hash"))
    .sort("_hash")
    .limit(50_000)
    .drop("_hash")
    .collect()
)

print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

---
## 2. Data Quality Assessment

### 2.1 Schema Overview

In [ ]:
print(f"Total columns: {len(df.schema)}\n")
for col_name, col_type in df.collect_schema().items():
    print(f"  {col_name}: {col_type}")

### 2.2 Primary Key Verification

The `code` column is the product barcode — expected to be the primary key.

In [ ]:
is_unique = df["code"].n_unique() == df.height
has_nulls = df["code"].null_count() > 0

print(f"code is unique:    {is_unique}")
print(f"code has nulls:    {has_nulls}")
print(f"Unique values:     {df['code'].n_unique():,}")

### 2.3 Duplicates Check

In [ ]:
n_duplicates = df.is_duplicated().sum()
print(f"Duplicate rows: {n_duplicates}")

### 2.4 Null Analysis

In [ ]:
# Raw null counts
df.null_count()

In [ ]:
# Null ratio per column (percentage)
null_ratio = df.select(
    ((pl.all().null_count() / df.height) * 100).name.suffix("_null_ratio")
)
print(null_ratio)

In [ ]:
# Columns with more than 80% nulls — candidates for dropping
high_null = (
    null_ratio
    .transpose(include_header=True, header_name="column")
    .sort("column_0", descending=True)
    .filter(pl.col("column_0") > 80)
)

print(f"Columns with >80% nulls: {high_null.shape[0]}")
print(high_null)

---
## 3. Data Profiling

### 3.1 Numeric Columns

**How to read the describe() output:**
- `count` — number of non-null values
- `null_count` — number of nulls
- `mean` — average value
- `std` — standard deviation (how spread out the values are)
- `min / max` — smallest and largest values
- `25%, 50%, 75%` — quartiles (25% of values are below the 25% number, etc.)

**Key rule:** If `std` is close to or larger than `mean`, there is high variation or outliers.

In [ ]:
numeric_cols = df.select(cs.numeric()).columns
print(f"Numeric columns: {len(numeric_cols)}")
print(numeric_cols)

In [ ]:
df.select(cs.numeric()).describe()

**Findings from numeric analysis:**

| Column | Issue | Action |
|--------|-------|--------|
| `completeness` | max = 1.1 (should be ≤ 1.0) | Investigate outlier |
| `nutriscore_score` | min = -17, high std | Normal range is -15 to 40 |
| `ecoscore_score` | 93% null | Drop in dbt |
| `with_sweeteners` | 99% null | Drop in dbt |
| `with_non_nutritive_sweeteners` | 96% null | Drop in dbt |
| `nova_group`, `nutriscore_score` | ~14% null | Keep null (no score = truly unknown) |
| `additives_n`, `ingredients_n` | ~3% null | Fill with 0 in dbt |
| `scans_n`, `unique_scans_n` | ~81% null | Fill with 0 in dbt |

### 3.2 String Columns

In [ ]:
string_cols = df.select(cs.string()).columns
print(f"String columns: {len(string_cols)}")
print(string_cols)

In [ ]:
df.select(cs.string()).describe()

In [ ]:
# Investigate compared_to_category format issue
print("compared_to_category sample:")
print(df["compared_to_category"].head(10))

**Findings from string analysis:**

| Column | Issue | Action in dbt |
|--------|-------|---------------|
| `compared_to_category` | Has `en:` prefix + `"en:null"` values | Strip prefix, convert `en:null` → null |
| `brands`, `quantity`, `origins` | Empty strings `""` present | Convert `""` → null |
| `nutriscore_grade`, `ecoscore_grade` | `"unknown"` string instead of null | Convert `"unknown"` → null |
| `nova_groups` | Duplicate of `nova_group` (int) | Drop in dbt |
| `creator`, `last_editor`, `last_modified_by` | Administrative, not analytical | Drop in dbt |
| `ecoscore_data` | JSON string, not a date | Parse as JSON in dbt if needed |

### 3.3 Complex Columns (List & Struct)

**Why these need special treatment in OLAP:**  
OLAP databases (like DuckDB) work best with flat tables — one value per cell.  
List and Struct columns must be either:
- **Exploded** into separate Dimension Tables (many-to-one relationship)
- **Flattened** into individual columns (one value per column)

In [ ]:
list_cols = [
    col for col, dtype in df.schema.items()
    if isinstance(dtype, pl.List)
]
print(f"List columns: {len(list_cols)}")
print(list_cols)

In [ ]:
struct_cols = [
    col for col, dtype in df.schema.items()
    if isinstance(dtype, pl.Struct)
]
print(f"Struct columns: {len(struct_cols)}")
print(struct_cols)

In [ ]:
# Explore nutriments structure
print("nutriments sample (3 rows):")
print(df["nutriments"].head(3))

In [ ]:
# Most common nutriment names across all products
nutriment_names = (
    df["nutriments"]
    .explode()
    .struct.field("name")
    .drop_nulls()
    .value_counts()
    .sort("count", descending=True)
    .head(20)
)
print(nutriment_names)

**List & Struct column decisions:**

| Column | Type | Action in dbt |
|--------|------|---------------|
| `allergens_tags` | List | Explode → `dim_allergens` |
| `ingredients_tags` | List | Explode → `dim_ingredients` |
| `categories_tags` | List | Explode → `dim_categories` |
| `countries_tags` | List | Explode → `dim_countries` |
| `additives_tags` | List | Explode → `dim_additives` |
| `traces_tags` | List | Explode → `dim_traces` |
| `food_groups_tags` | List | Explode → `dim_food_groups` |
| `packaging_tags` | List | Explode → `dim_packaging` |
| `origins_tags` | List | Explode → `dim_origins` |
| `packagings` | List[Struct] | Explode → `dim_packagings` |
| `nutriments` | List[Struct] | Flatten → 14 separate columns |
| `nutrient_levels_tags` | List | Flatten → single value |
| `categories_properties` | Struct | Unnest → merge into `dim_categories` |
| `images`, `editors`, `photographers` | List | Drop (not analytical) |

---
## 4. Canadian Products Analysis

This project targets the Canadian food database.  
We need to understand how many Canadian products exist in our sample.

In [ ]:
canada = df.filter(
    pl.col("countries_tags").list.contains("en:canada")
)

print(f"Total products in sample:   {df.shape[0]:,}")
print(f"Canadian products:          {canada.shape[0]:,}")
print(f"Percentage:                 {canada.shape[0] / df.shape[0] * 100:.1f}%")

**⚠️ Important Finding:**  
Only **~0.3%** of our sample are Canadian products.

This is a critical issue for the project because:
- Our EDA is based mostly on non-Canadian products
- The OLAP database should focus on Canadian data

**This will be addressed in the dlt pipeline** by loading from `ca.openfoodfacts.org` directly,
which contains only Canadian products.

---
## 5. Core Analytical Columns

Based on the EDA above, these are the columns selected for the Canadian Food OLAP schema.  
All other columns will be dropped or transformed in dbt.

In [ ]:
core_columns = [
    # Product Identity
    "code",            # Primary key (barcode)
    "product_name",    # List[Struct] — will be flattened to main language
    "brands",          # Brand name
    "lang",            # Primary language
    "quantity",        # Product quantity (e.g. '350 g')
    "serving_size",    # Serving size
    "created_t",       # Unix timestamp — creation date
    "last_modified_t", # Unix timestamp — last update

    # Classification
    "categories",      # Raw categories string
    "countries_tags",  # List — will explode → dim_countries
    "food_groups_tags",# List — will explode → dim_food_groups

    # Nutrition Scores
    "nutriscore_grade",  # Letter grade a-e (keep null if unknown)
    "nutriscore_score",  # Numeric score -15 to 40 (keep null if unknown)
    "nova_group",        # 1-4 processing level (keep null if unknown)
    "ecoscore_grade",    # Environmental score
    "nutrient_levels_tags", # List of nutrient level labels

    # Ingredients
    "ingredients",           # Raw ingredients string
    "allergens_tags",        # List — will explode → dim_allergens
    "traces_tags",           # List — will explode → dim_traces
    "additives_n",           # Count of additives (fill null → 0)
    "additives_tags",        # List — will explode → dim_additives
    "ingredients_analysis_tags", # List — vegan/palm oil analysis
    "ingredients_tags",      # List — will explode → dim_ingredients

    # Nutriments (will be flattened into 14 columns in dbt)
    "nutriments",

    # Popularity
    "popularity_key",  # Scan-based popularity score
    "obsolete",        # Boolean — is product discontinued?
]

df_core = df.select(core_columns)

print(f"Core columns selected: {len(core_columns)}")
print(f"df_core shape: {df_core.shape}")

In [ ]:
print(df_core.describe())

In [ ]:
# Null counts in core columns
core_null = df_core.select(
    ((pl.all().null_count() / df_core.height) * 100).name.suffix("_null%")
)
print(core_null)

### Nutriments Flattening Plan

The `nutriments` column is `list[struct[9]]` — each product has a different set of nutrients.
This structure cannot be efficiently queried in DuckDB.

**Solution:** Extract the 14 most common and nutritionally relevant nutrients into separate columns.

This allows analytical queries like:
```sql
SELECT AVG(fat_100g), AVG(sugars_100g) 
FROM fact_products 
WHERE nova_group = 4
```

In [ ]:
core_nutrients = [
    "energy-kcal",   # Calories
    "fat",           # Total fat
    "saturated-fat", # Saturated fat
    "trans-fat",     # Trans fat — Canada was first country to ban it
    "carbohydrates", # Total carbohydrates
    "sugars",        # Sugars
    "proteins",      # Proteins
    "salt",          # Salt
    "sodium",        # Sodium
    "fiber",         # Dietary fiber
    "calcium",       # Calcium
    "iron",          # Iron
    "vitamin-c",     # Vitamin C
    "vitamin-a",     # Vitamin A
]

print(f"Core nutrients selected: {len(core_nutrients)}")
print(core_nutrients)

---
## 6. ELT Action Plan

Summary of all decisions made during EDA.  
These will be implemented in the dbt transformation layer.

---

### 6.1 Columns to Drop (in dbt)

**High null ratio (>80%) — 27 columns**  
Identified via `null_ratio` analysis above.

**Administrative columns — not useful for analysis**
```
checkers_tags, correctors_tags, informers_tags
creator, last_editor, last_modified_by
entry_dates_tags, last_edit_dates_tags
rev, schema_version, max_imgid
data_quality_errors_tags, data_quality_info_tags
data_quality_warnings_tags, data_sources_tags
misc_tags, states_tags, complete, completeness
```

**Duplicate columns — info already exists elsewhere**
```
brands_tags          → duplicate of brands
nova_groups_tags     → duplicate of nova_group (int)
nova_groups          → duplicate of nova_group (int)
main_countries_tags  → duplicate of countries_tags
languages_tags       → duplicate of lang
product_quantity     → duplicate of quantity
product_quantity_unit→ duplicate of quantity
ingredients_text     → duplicate of ingredients
ingredients_original_tags → duplicate of ingredients_tags
generic_name         → duplicate of product_name
```

**Low value columns**
```
with_sweeteners               → 99% null
with_non_nutritive_sweeteners → 96% null
ecoscore_score                → 93% null
images                        → not analytical
editors, photographers        → not analytical
```

---

### 6.2 String Transformations (in dbt)

| Column | Issue | Fix |
|--------|-------|-----|
| `compared_to_category` | `en:` prefix on all values | `str.replace('en:', '')` |
| `compared_to_category` | `'en:null'` string values | Convert to null |
| `brands`, `quantity`, `origins` | Empty string `''` | Convert to null |
| `nutriscore_grade`, `ecoscore_grade` | `'unknown'` string | Convert to null |
| `created_t`, `last_modified_t` | Unix timestamp (Int64) | Convert to datetime |

---

### 6.3 Null Strategy (in dbt)

| Column(s) | Strategy | Reason |
|-----------|----------|--------|
| `nova_group`, `nutriscore_score` | Keep null | null = product has no score, filling would be misleading |
| `additives_n`, `ingredients_n` | Fill with 0 | null here likely means 0 (not measured) |
| `ingredients_from_palm_oil_n` | Fill with 0 | Same reasoning |
| `scans_n`, `unique_scans_n` | Fill with 0 | No scans recorded = 0 scans |

---

### 6.4 Nutriments Flattening (in dbt)

Extract from `list[struct[9]]` into 14 flat columns:

```
energy_kcal_100g, fat_100g, saturated_fat_100g, trans_fat_100g,
carbohydrates_100g, sugars_100g, proteins_100g, salt_100g,
sodium_100g, fiber_100g, calcium_100g, iron_100g,
vitamin_c_100g, vitamin_a_100g
```

---

### 6.5 Dimension Tables to Create (in dbt)

| Dimension Table | Source Column | Method |
|-----------------|---------------|--------|
| `dim_allergens` | `allergens_tags` | Explode list |
| `dim_ingredients` | `ingredients_tags` | Explode list |
| `dim_categories` | `categories_tags` + `categories_properties` | Explode + unnest |
| `dim_countries` | `countries_tags` | Explode list |
| `dim_additives` | `additives_tags` | Explode list |
| `dim_traces` | `traces_tags` | Explode list |
| `dim_food_groups` | `food_groups_tags` | Explode list |
| `dim_packaging` | `packaging_tags` | Explode list |
| `dim_origins` | `origins_tags` | Explode list |
| `dim_packagings` | `packagings` | Explode list[struct] |

---

### 6.6 Data Enrichment (in dlt)

- **Missing brands:** Can be fetched from the OFF API using `code` (barcode)
  ```
  GET https://world.openfoodfacts.org/api/v2/product/{barcode}
  ```
- **Canadian products:** Load directly from `ca.openfoodfacts.org` to get full Canadian dataset  
  (only 0.3% of the global sample are Canadian products)

---

### 6.7 Open Questions (for mentor review)

1. Should we keep the full OFF schema or only the 25 core columns?
2. Should `nutriments` be flattened into columns or kept as a separate `dim_nutriments` table?
3. How to handle the 0.3% Canadian coverage in the global sample — use `ca.openfoodfacts.org` exclusively?
4. Priority for data enrichment: which missing fields are most critical to fill via API?